# Bellwether — strategy detection

Accounts are what Polymarket exposes; **strategies** are the durable signal — a
behavioral template run over and over: across markets in a recurring family
(scalping `btc-updown-5m-*`), across accounts (same archetype again and again),
or as copy/follow chains.

1. behavioral features → 2. explainable archetype → 3. repeated templates →
4. lead-lag copy chains.

In [ ]:
from bellwether_analytics.core import load_events_df, load_trades_df
from bellwether_analytics.strategy import (
    classify,
    detect_followers,
    extract_features,
    strategy_templates,
)

trades = load_trades_df(platform="polymarket")
events = load_events_df(platform="polymarket")
feats = extract_features(trades, events)
labeled = classify(feats)
labeled[["archetype", "trades_per_day", "net_direction", "roundtrip_ratio",
         "split_merge_ratio", "top_family", "top_family_share", "reason"]]

In [ ]:
# Strategies implemented over and over: (archetype, market-family) across accounts.
strategy_templates(trades, labeled["archetype"], min_wallets=2)

In [ ]:
# Copy/follow chains: wallets that systematically trade just after another.
detect_followers(trades, max_lag_seconds=120, min_events=3)

## Temporal recurrence (same cycle, over and over in time)
Does a wallet run the same buy→accumulate→redeem cycle on a regular cadence?
High regularity + many completed cycles = a templated strategy run repeatedly.

In [ ]:
import pandas as pd
from bellwether_analytics.strategy import recurrence_in_time_report

rows = [recurrence_in_time_report(trades, events, w) for w in trades['wallet'].unique()[:5]]
pd.DataFrame(rows)[['wallet','top_family','is_recurring','regularity','n_cycles',
                    'dominant_period_seconds','cycle_consistency']]

## Strategy -> profitability (templates, not accounts)
Rank (archetype, market-family) templates by strictly out-of-sample realized
P&L. The shuffled-label control must show no edge.

In [ ]:
from bellwether_analytics.strategy import rank_templates

split = '2026-05-01'
ranked = rank_templates(trades, split_ts=split)
shuffled = rank_templates(trades, split_ts=split, shuffle=True, seed=0)
display(ranked.head(15))
print('shuffled-control top median P&L:', None if shuffled.empty else shuffled.iloc[0]['median_pnl'])

## Real-data trackability (Experiment A from real followers)
For significant copy-chains, what did real followers actually capture?

In [ ]:
from bellwether_analytics.experiments import rank_leaders
from bellwether_analytics.strategy import detect_followers_significant

pairs = detect_followers_significant(trades, max_lag_seconds=120, min_events=3)
rank_leaders(trades, pairs)  # per-leader empirical copyability + captured edge

## Unsupervised cross-check of the rule archetypes (diagnostic)
Cluster the feature vectors; compare to rule labels; surface disagreements.

In [ ]:
from bellwether_analytics.strategy import cluster_wallets, compare_to_rules, recommend_thresholds

clusters = cluster_wallets(feats, k=3)
cmp = compare_to_rules(clusters, labeled['archetype'])
print('agreement:', cmp['agreement'])
display(cmp['contingency'])
recommend_thresholds(feats, cmp)